Report Capstone Project

Tobias van der Meer, Rafael Lima Araujo Rego, Janniek Graveland, Jorrit Paques and Bram Verlaat

In this report, we will briefly explain the architectures of both the CNN model and UNet model. Next to giving an explanation, we will also show the code of the architecture of our models. The two best models are explained in detail and all the codes of the other model architectures care just shown and named below. This notebook is not runable as it shows only parts of our code that shows the model's architecture. The whole code is sent in an other file.

The best CNN-model:
The best performing convolutionial neural network model is the one that starts with a convolutional part, which is very similar to the encoder part of a u-net, and ends with three fully connected layers. It takes a single-channel 2D-input and outputs a 60x60 2D field. It It gives a MAE of 1.053 when trained, and it generates a smooth output field which we will show below. 






In [ ]:
class Model(nn.Module):
    # During one of the meetings the idea of putting a fully connected layer behind a convolutional one came up.
    # This model starts with a convolutional part that is very similar to the encoder part of a u-net, after that
    # it has three fully connected layers. This model works very well, giving a MEA of 1.053 when trained. The model
    # also generates a relatively smooth output field and it does not have bad outliers

    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()
        self.conv1 = nn.Conv2d(1, 16, 5, padding='same', padding_mode='reflect')
        self.conv2 = nn.Conv2d(16, 16, 7, padding='same', padding_mode='zeros')
        self.pool1 = nn.MaxPool2d(2)  # 30x30
        self.conv3 = nn.Conv2d(16, 32, 5, padding=3, padding_mode='zeros')  # 32x32
        self.conv4 = nn.Conv2d(32, 32, 5, padding='same', padding_mode='zeros')
        self.pool2 = nn.MaxPool2d(2)  # 16x16
        self.conv5 = nn.Conv2d(32, 64, 5, padding='same', padding_mode='zeros')
        self.conv6 = nn.Conv2d(64, 64, 5, padding='same', padding_mode='zeros')
        self.pool3 = nn.MaxPool2d(2) # 8x8
        self.conv7 = nn.Conv2d(64, 64, 5, padding='same', padding_mode='zeros')
        self.conv8 = nn.Conv2d(64, 64, 3, padding='same', padding_mode='zeros')

        self.fc1 = nn.Linear(4096, 2048)
        self.fc2 = nn.Linear(2048, 2048)
        self.fc3 = nn.Linear(2048, 3600)



    def forward(self, x):
        #convolutional part
        h = self.relu(self.conv1(x))
        h = self.relu(self.conv2(h))
        h = self.pool1(h)

        h = self.relu(self.conv3(h))
        h = self.relu(self.conv4(h))
        h = self.pool2(h)

        h = self.relu(self.conv5(h))
        h = self.relu(self.conv6(h))
        h = self.pool3(h)

        h = self.relu(self.conv7(h))
        h = self.relu(self.conv8(h)).view(-1, 4096)

        # fully connected part
        h = self.relu(self.fc1(h))
        h = self.relu(self.fc2(h))
        h = self.fc3(h)

        return h.reshape((-1, 60, 60))

The best Unet-model:
The best Unet-model is specifically for a 64x64 grid for which it is possible. It is built up with an encoder-bottleneck-decoder structure. It encodes local features and compresses this to the bottleneck. The key part of this model architecture is the so-called "global brain". Where a standard Unet often fails on global linear gradients, this model takes into account these global features. 



In [ ]:
class Model(nn.Module):
    """
    U-Net specifically for 60x60 grid.
    Includes the 'Global Brain' fix in the bottleneck to catch that linear gradient.
    """

    def __init__(self, base_ch: int = 32, enforce_dirichlet_row0: bool = False):
        super().__init__()

        self.enforce_dirichlet_row0 = enforce_dirichlet_row0

        # Encoder path
        # -----------------------
        # Input is 1 channel (logK map)
        in_ch = 1

        # Level 1: 60x60
        self.enc1 = ConvBlock(in_ch, base_ch)
        self.pool1 = nn.MaxPool2d(2)  # drops to 30x30

        # Level 2: 30x30
        self.enc2 = ConvBlock(base_ch, 2 * base_ch)
        self.pool2 = nn.MaxPool2d(2)  # drops to 15x15

        # Bottleneck: 15x15
        self.center = ConvBlock(2 * base_ch, 4 * base_ch)

        # -----------------------
        # THE FIX: Global Information Injection
        # Problem: Standard U-Net sucks at learning the global linear gradient (gravity).
        # Solution: Squeeze everything to 1x1, learn the bias, add it back.
        # -----------------------

        # 1. Squash 15x15 spatial dims to 1x1.
        # Prevents parameter explosion (keeps params ~65k instead of 14M).
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        feature_ch = 4 * base_ch  # e.g. 256 channels

        # 2. MLP to figure out the global slope/offset
        self.global_dense = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_ch, feature_ch),
            nn.ReLU(inplace=True),
            nn.Linear(feature_ch, feature_ch),
            nn.Unflatten(1, (feature_ch, 1, 1))  # Reshape to (N, C, 1, 1) for broadcasting
        )

        # Decoder path
        # -----------------------

        # Level 2 Up: 15 -> 30
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec2 = ConvBlock(4 * base_ch + 2 * base_ch, 2 * base_ch)

        # Level 1 Up: 30 -> 60
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec1 = ConvBlock(2 * base_ch + base_ch, base_ch)

        # Output projection
        self.out = nn.Conv2d(base_ch, 1, kernel_size=1)

        # Hardcoded boundary value for top row (physically h=100m, normalized)
        # y = (h - 146) / 37
        self.dirichlet_row0_value = (100.0 - 145.3243) / 35.5957

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (N, 1, 60, 60)

        # --- Encoder ---
        x1 = self.enc1(x)  # 60x60
        x2 = self.enc2(self.pool1(x1))  # 30x30

        # --- Bottleneck ---
        x_center = self.pool2(x2)  # 15x15
        x_center = self.center(x_center)

        # --- Global Injection ---
        # 1. Get average state of the system
        global_feat = self.global_pool(x_center)
        # 2. Process through dense layer
        global_feat = self.global_dense(global_feat)
        # 3. Add back to feature map (Residual connection)
        # Broadcasting happens here: (N, C, 15, 15) + (N, C, 1, 1)
        x_center = x_center + global_feat

        # --- Decoder ---
        d2 = self.up2(x_center)
        d2 = torch.cat([d2, x2], dim=1)  # Skip connection
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, x1], dim=1)  # Skip connection
        d1 = self.dec1(d1)

        out = self.out(d1)

        # Force physics on the top row if flag is set
        if self.enforce_dirichlet_row0:
            out[:, :, 0, :] = self.dirichlet_row0_value

        return out


Other CNN-models: